# 03. Graph RAG with LangChain

[NB1](./01_introduction.ipynb) was the crawl. [NB2](./02_querying_the_graph.ipynb) was the walk. **This is the run.**

We wire an LLM in. The job: take a natural-language question, translate it to Cypher, run it on the graph, and synthesize an answer from the result. Same four-box pipeline as module 01, but the retriever is openCypher instead of similarity search.

This notebook does the job two ways. **First by hand, then with `KuzuQAChain` from LangChain.** Same answer, very different lines of code. The point isn't to pick a side. It's to look at exactly what the chain is wrapping, so you can choose when to use it and when to stay on the manual loop.

## The loop, laid out

```
   "Which directors has               ┌──────────────────────┐
    Florence Pugh worked under?"      │  LLM (writes Cypher) │
          │                           └──────────────────────┘
          ▼                                       │
   ┌─────────────┐                                ▼
   │   Query     │            MATCH (florence)-[:ACTED_IN]->(m)
   └─────────────┘                  <-[:DIRECTED]-(d) RETURN d.name
                                                │
                                                ▼
                              ┌────────────────────────────────┐
                              │   conn.execute(cypher)         │
                              │   -> a pandas DataFrame        │
                              └────────────────────────────────┘
                                                │
                                                ▼
                              ┌────────────────────────────────┐
                              │   LLM (synthesizes answer)     │
                              │   sees question + result rows  │
                              └────────────────────────────────┘
                                                │
                                                ▼
                              "Florence Pugh has worked under
                               Christopher Nolan, Denis Villeneuve,
                               and Greta Gerwig."
```

Two LLM calls per question: one to write the Cypher, one to turn the result into prose. Hold onto that, it's worth noticing again when we compare the manual loop to the chain.

## Setup

> **Run this notebook top to bottom.** Each cell depends on variables (`conn`, `client`, `schema`) defined by the cells above it. If you jump straight to a code cell and see `NameError: name 'client' is not defined`, run this Setup cell first (Kernel → Restart & Run All Cells does the whole thing at once).

NB3 needs an OpenAI key. Copy `.env.example` to `.env` and paste yours.

We also `build_if_missing()` to make sure the movie graph is on disk.

In [ ]:
from helpers import load_env, get_openai_client, get_kuzu_conn
from scripts.build_corpus import build_if_missing

build_if_missing()

cfg = load_env()
client = get_openai_client(cfg)
conn = get_kuzu_conn()

print("Connected. Chat model:", cfg.openai_chat_model)

## Act I: the manual loop

Before we hand the keys to LangChain, let's build it by hand. Four steps:

1. Grab the schema as a string. The LLM needs to know what node and relationship types exist.
2. Send the question + schema to the LLM, ask for Cypher.
3. Run the Cypher on Kuzu.
4. Send the question + result rows back to the LLM, ask for an answer.

This is the whole RAG loop. Everything fancier is an elaboration of it.

### Step 1: the schema

We could write a schema string by hand, but the official `langchain-kuzu` package already has a `KuzuGraph` class that builds one from your live database. Borrowing just the schema generation from it (not the chain) gives us the same input the chain will get later, which makes the comparison fair.

In [ ]:
from langchain_kuzu.graphs.kuzu_graph import KuzuGraph

# Reuse the existing connection's Database object instead of opening a new one.
# Kuzu locks the database file: a second kuzu.Database(...) call against the
# same path from the same process will fail with "Could not set lock on file".
# allow_dangerous_requests=True is required because KuzuGraph can execute
# arbitrary Cypher. Fine here: it's our local DB.
graph = KuzuGraph(conn.database, allow_dangerous_requests=True)
schema = graph.schema
print(schema)

### Step 2: ask the LLM to write Cypher

We prompt with the schema and the question, and we keep the instructions tight. The prompt is borrowed in shape from what LangChain uses internally (see the `langchain_kuzu.chains.graph_qa.prompts` module): tell the model exactly what to produce, give it the schema, give it the question.

In [ ]:
CYPHER_INSTRUCTIONS = """You are an expert in translating natural language questions into Cypher statements.
You will be provided with a question and a graph schema.
Use only the provided relationship types and properties in the schema to generate a Cypher statement.
The Cypher statement could retrieve nodes, relationships, or both.
Do not include any explanations or apologies in your responses.
Output ONLY the Cypher statement."""


# Friendly guard: clearer than NameError deep inside the function below
for _required in ("client", "cfg", "schema"):
    if _required not in globals():
        raise NameError(
            f"`{_required}` is not defined. Did you skip the Setup cell? "
            "Run Kernel -> Restart & Run All Cells from the top of the notebook."
        )


def generate_cypher(question: str) -> str:
    # Plain concatenation, not .format(). The schema string often contains
    # literal '{' and '}' characters (e.g. node property descriptors) which
    # would otherwise be parsed as format placeholders and crash.
    prompt = (
        CYPHER_INSTRUCTIONS
        + "\n\nSchema:\n" + schema
        + "\n\nThe question is:\n" + question
    )
    resp = client.chat.completions.create(
        model=cfg.openai_chat_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    cypher = resp.choices[0].message.content.strip()
    # The model sometimes wraps output in ```cypher ... ``` fences. Strip them.
    if cypher.startswith("```"):
        cypher = cypher.split("```")[1]
        if cypher.startswith("cypher"):
            cypher = cypher[len("cypher"):]
        cypher = cypher.strip()
    return cypher


question = "Which directors has Florence Pugh worked under?"
cypher = generate_cypher(question)
print(cypher)

### Step 3: run it

In [ ]:
rows = conn.execute(cypher).get_as_df()
rows

### Step 4: synthesize an answer

Hand the question, the rows, and the model: ask for a sentence. `temperature=0` keeps it from creatively wandering off the data.

In [ ]:
ANSWER_INSTRUCTIONS = """You are answering a question using only the data provided.
Do not make anything up. If the data doesn't answer the question, say so."""


def synthesize_answer(question: str, rows) -> str:
    # Plain concatenation, same reason as generate_cypher: the rows table
    # can contain '{' / '}' which would break .format().
    prompt = (
        ANSWER_INSTRUCTIONS
        + "\n\nQuestion: " + question
        + "\n\nData (as a table):\n" + rows.to_string(index=False)
        + "\n\nAnswer:"
    )
    resp = client.chat.completions.create(
        model=cfg.openai_chat_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()


print(synthesize_answer(question, rows))

### Wrap it in one function

So we can ask other questions with one call.

In [ ]:
def manual_graph_rag(question: str) -> dict:
    cypher = generate_cypher(question)
    rows = conn.execute(cypher).get_as_df()
    answer = synthesize_answer(question, rows)
    return {"question": question, "cypher": cypher, "rows": rows, "answer": answer}


result = manual_graph_rag("How many films has Jordan Peele directed?")
print("Cypher generated:")
print(result["cypher"])
print()
print("Rows:")
print(result["rows"])
print()
print("Answer:")
print(result["answer"])

## Act II: the same thing with `KuzuQAChain`

Same job. One import, one chain, one call.

`KuzuQAChain.from_llm(...)` builds the orchestration for you. Internally it's still doing exactly what we just did by hand: schema injection, Cypher generation, query execution, answer synthesis. Two LLM calls. Same shape.

In [ ]:
from langchain_kuzu.chains.graph_qa.kuzu import KuzuQAChain
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=cfg.openai_chat_model, temperature=0, api_key=cfg.openai_api_key)

chain = KuzuQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True,
)

answer = chain.invoke({"query": "Which directors has Florence Pugh worked under?"})
print()
print("Final answer:", answer["result"])

## What the chain bought you, and what it hid

**Bought you:** about fifteen lines of orchestration code. It wired the two LLM calls, formatted the schema, parsed the model's Cypher output, ran it on the graph, formatted the rows for the answer prompt, and stitched the result back. Saved you the effort of writing all that. Worth it on day fifty when you're building three more of these.

**Hid:** the prompts. The chain ships with default `cypher_prompt` and `qa_prompt` templates that are reasonable but invisible. The first time the LLM writes wrong Cypher (and it will, eventually), debugging means going into the LangChain source to see what the prompt actually looked like. With the manual loop you already know.

**Cost it doesn't tell you about:** every question is two LLM calls. The first to write Cypher, the second to write the answer. That's fine at one cent per question; less fine when you scale to a million questions a day and discover half the cost was the answer-synthesis step, and you could have skipped it for some queries.

**The deeper point:** the chain is a fine abstraction. It's just an abstraction over something simple enough that you can write it yourself in fifty lines. Build the manual loop first so you can see the seams. Then if a chain saves you time, use the chain. The order matters.

## Act III: the question vector search can't answer

This is why graph RAG exists in the first place.

> *"Which directors have worked with actors who appeared in films produced by Warner Bros?"*

A vector retriever would embed that question and look for similar text. There's no single document in our corpus that says "Christopher Nolan worked with an actor who was in a WB film" because that statement isn't a fact about any one person, it's a path through three.

The graph retriever can answer it directly. Five hops:

```
Studio <- Movie <- Person -> Movie <- Person
   WB       (a)    (actor)   (b)    (director)
```

Note the actor binds to one Person node and traverses out through it. Note also that we exclude `b = a` so the answer doesn't include the WB film itself (otherwise every actor "worked with" their own director through their own film, which is true but trivial).

In [ ]:
multi_hop_question = (
    "Which directors have worked with actors who appeared in films produced by Warner Bros? "
    "List the director, the shared actor, the WB film, and the other film. "
    "Exclude cases where the two films are the same."
)

result = manual_graph_rag(multi_hop_question)
print("Cypher generated:")
print(result["cypher"])
print()
print("Rows:")
print(result["rows"])
print()
print("Answer:")
print(result["answer"])

### What graph RAG doesn't fix

Honest closing: this is not a silver bullet.

- **Hallucinated Cypher.** The LLM writes Cypher that looks valid but references a label that doesn't exist, or gets the relationship direction backwards. Always wrap in `try/except` in production. Surface the error to the model and ask it to try again.
- **Schema brittleness.** Rename a relationship and every prompt that mentions it is stale. The chain's schema injection updates automatically, but any custom prompt or fine-tuned model doesn't.
- **No fuzzy entity matching.** "Florence Pugh" matches, "florence pug" does not. For real apps you bolt on an entity-linking step (often a vector lookup over node names) before generating Cypher.
- **Two LLM calls per question.** Cost adds up. For high-volume use cases you can drop the answer-synthesis step and return the raw rows.
- **Cold-start brittleness.** A small graph means the LLM has to be perfect. Production graph RAG runs on graphs with thousands of relationship types and millions of nodes, where the schema injection becomes a real prompt-engineering challenge.

Modules 04 and 05 of this repo cover evaluation and advanced patterns that address most of these.

## Appendix: switching to Neo4j

Same code, two import lines change. LangChain has a `Neo4jGraph` class with the same `schema` property and a `Neo4jQAChain` with the same `from_llm` signature.

```python
# Was:
from langchain_kuzu.graphs.kuzu_graph import KuzuGraph
from langchain_kuzu.chains.graph_qa.kuzu import KuzuQAChain

graph = KuzuGraph(kuzu.Database("data/movies.kuzu"), allow_dangerous_requests=True)

# Becomes:
from langchain_neo4j import Neo4jGraph
from langchain_neo4j.chains.graph_qa.cypher import GraphCypherQAChain

graph = Neo4jGraph(
    url="bolt://localhost:7687",
    username="neo4j",
    password="...",
)
```

The Cypher you write doesn't change. openCypher is the bridge.

Why we used Kuzu by default: no Docker, no signup, no cloud instance to provision. The `data/movies.kuzu` file is portable; you can ship it with the repo (we don't, because it's gitignored, but you could). For production with concurrent writes or a graph bigger than RAM, Neo4j is the more battle-tested choice.

## Recap

| Step | Manual loop | Chain |
|---|---|---|
| Generate Cypher | One LLM call with schema + question | `cypher_generation_chain` |
| Execute | `conn.execute(cypher).get_as_df()` | `graph.query(cypher)` |
| Synthesize answer | One LLM call with question + rows | `qa_chain` |
| Total lines | ~40 | ~5 |
| Total LLM calls per question | 2 | 2 |
| Visible prompt | Yes | Hidden (in `langchain_kuzu.chains.graph_qa.prompts`) |

Both work. The chain is faster to write. The manual loop is easier to debug.

## A gap worth naming before you go

Look back at what `manual_graph_rag` returns. The "answer" is built from rows of structured data: names, titles, numbers. That's a perfectly valid form of retrieval, but it isn't what most people mean by "RAG" in production. Production graph RAG attaches **real text** to nodes (plot summaries, script excerpts, biographies) and the graph tells you *which* texts to retrieve. The graph is the index over the text, not a replacement for it.

**NB4 picks this up.** It's the natural follow-up to everything you just learned, and it's coming next in this module. Same Cypher-first pattern, but the nodes carry long-form text (real movie scripts), and the answer-synthesis step has actual prose to ground itself in. That's the canonical production pattern.

## Where to go next in the repo

- **[Module 03: Vectorless RAG.](../03-vectorless-rag/)** Keyword search and BM25, for the times you don't need vectors *or* a graph.
- **[Module 04: Evaluating RAG.](../04-evaluating-rag/)** How to tell whether your graph RAG actually got better. Both modules 01 and 02 punted on this.
- **[Module 05: Advanced RAG.](../05-advanced-rag/)** Reranking, query rewriting, hybrid (vector + graph) retrieval. Where the script-chunking pattern teased above gets a full treatment.